# Spatial Kernel Coupling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML

from kuramoto.config import (SimulationConfig, GridConfig,
    CouplingConfig, NeuronConfig, InitThetaConfig, InitOmegaConfig, build_simulation
)
from kuramoto.coupling import CouplingMatrix
from kuramoto.kernels import (
    gaussian_kernel,
    exponential_kernel,
    step_kernel,
    mexican_hat_kernel,
    gabor_kernel,
    elongated_gaussian_kernel,
    lesioned_wedge_kernel, 
)
from kuramoto.grid import CorticalGrid
from kuramoto.analysis import get_R, local_order
from kuramoto.plotting import plot_2d, plot_coupling_matrix, coupling_to_2d

%matplotlib inline

In [ ]:
# Higher level sim params
SEED = 42
T_END = 40.0
dt = 0.1
RNG = np.random.seed(SEED)

base_K = 1.0 # Base coupling strength
kernel_max_r = 8.0 # Max kernel extents

# Grid spec
grid_cfg = GridConfig(shape=(32, 32))

# State and omega initialization
init_theta = InitThetaConfig(mode="uniform")
init_omega = InitOmegaConfig(mode="normal", mu=0, sigma=1.0)

## 1. Kernel Visualization

### Symmetric Kernel profiles

In [ ]:
d = np.linspace(0.01, 8.0, 200)

kernels_1d = {
    "Gaussian (sigma=2)": gaussian_kernel(d, sigma=2.0),
    "Exponential (tau=2)": exponential_kernel(d, tau=2.0),
    "Step (radius=3)": step_kernel(d, radius=3.0),
    "Mexican Hat (sigma_e=1, sigma_i=3)": mexican_hat_kernel(d, sigma_e=1.0, sigma_i=3.0),
}

fig, ax = plt.subplots(figsize=(10, 5))
for label, w in kernels_1d.items():
    ax.plot(d, w, linewidth=2, label=label)
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax.set_xlabel("Distance")
ax.set_ylabel("Kernel weight")
ax.set_title("Spatial Coupling Kernels -- Weight vs Distance")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2D kernel maps

## Theta gradient analysis (traveling wave)

Post-process the θ field to get the spatial gradient ∇θ: **map** of |∇θ| (and optional direction) and **bulk metric** (e.g. mean |∇θ| over time). Run a simulation above first so `sim`, `t_list`, and `state_list` are in scope.

In [ ]:
gauss_params = {"sigma": 2.0}
exp_params = {"tau": 2.0}
mh_params = {"sigma_e": 1.0, "sigma_i": 3.0}
gabor_params = {"sigma": 2.0, "theta_pref": 0.0, "freq": 0.2}
elong_gauss_params = {"sigma_par": 3.0, "sigma_perp": 1.0, "theta": 0.0}
lw_params = {"theta_center": 0.0, "wedge_width": np.pi / 2, "sigma": 2.0}

kernel_configs = [
    ("Gaussian", "gaussian", gauss_params, kernel_max_r),
    ("Exponential", "exponential", exp_params, kernel_max_r),
    ("Step", "step", {}, kernel_max_r),
    ("Mexican Hat", "mexican_hat", mh_params, kernel_max_r),
    ("Gabor", "gabor", gabor_params, kernel_max_r),
    ("Elongated Gaussian", "elongated_gaussian", elong_gauss_params, kernel_max_r),
    ("Lesioned Wedge", "lesioned_wedge", lw_params, kernel_max_r),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.ravel()
for ax, (label, kernel_name, kp, radius) in zip(axes, kernel_configs):
    grid = CorticalGrid(shape=grid_cfg.shape, periodic_bc=grid_cfg.periodic)

    center_node_idx = grid.idx_2d_to_flat(row=15,col=15)

    cm = CouplingMatrix(grid, kernel=kernel_name, base_strength=base_K,
                        kernel_params=kp.copy(), radius=radius, mode="spatial")

    footprint = coupling_to_2d(cm.K, grid, mode="from", node=center_node_idx)
    
    im = ax.imshow(footprint, cmap="RdBu_r", aspect="equal", interpolation="nearest")
    ax.set_title(label)
    plt.colorbar(im, ax=ax, shrink=0.8)
axes[-1].axis("off")  # hide unused subplot
plt.suptitle("2D Coupling Footprint from Center Node", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 2. Dynamics Comparison

In [ ]:
kernel_sim_configs = {
    "Gaussian": CouplingConfig(mode="spatial", kernel="gaussian", base_strength=base_K, radius=kernel_max_r, kernel_params=gauss_params),
    "Exponential": CouplingConfig(mode="spatial", kernel="exponential", base_strength=base_K, radius=kernel_max_r, kernel_params=exp_params),
    "Step": CouplingConfig(mode="spatial", kernel="step", base_strength=base_K, radius=kernel_max_r,),
    "Mexican Hat": CouplingConfig(mode="spatial", kernel="mexican_hat", base_strength=base_K, radius=kernel_max_r, kernel_params=mh_params),
    "Gabor": CouplingConfig(mode="spatial", kernel="gabor", base_strength=base_K, radius=kernel_max_r, kernel_params=gabor_params),
    "Elongated Gaussian": CouplingConfig(mode="spatial", kernel="elongated_gaussian", base_strength=base_K, radius=kernel_max_r, kernel_params=elong_gauss_params),
    "Lesioned Wedge": CouplingConfig(mode="spatial", kernel="lesioned_wedge", base_strength=base_K, radius=kernel_max_r, kernel_params=lw_params),
}

results_kern = {}

for label, coup_cfg in kernel_sim_configs.items():
    # Create sim config
    cfg = SimulationConfig(
        grid=grid_cfg,
        coupling=coup_cfg,
        initial_theta=init_theta,
        initial_omega=init_omega,
        seed=SEED,
    )

    # Build sim
    sim = build_simulation(config=cfg, rng=RNG)

    # Run sim
    results = sim.run((0, T_END), dt, rng=RNG)
    t_list = results['ts'].tolist()
    state_list = [{'theta': th, 'theta_dot': dth, 'omega': results['omega']} for th, dth in zip(results['theta'], results['theta_dot'])]

    # Postprocess
    R_list, _ = get_R(results['theta'])
    local_R_list = local_order(results['theta'], sim.grid)
    
    results_kern[label] = {
        "t_list": t_list, 
        "state_list": state_list,
        "R": R_list,
        "local_R": local_R_list,
        "grid": sim.grid,
        "coupling": sim.coupling,
    }

    print(f"{label}: R_final = {R_list[-1]:.3f}")

In [ ]:
### Coupling Matrix Heatmaps
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, (label, res) in zip(np.ravel(axes), results_kern.items()):
    K = res["coupling"].K
    if K.nnz == 0:
        ax.set_title(f"{label}\n(uniform -- no matrix)")
        continue
    plot_coupling_matrix(K, ax=ax, title=label)
plt.suptitle("Full Coupling Matrices", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, res in results_kern.items():
    ax.plot(res["t_list"], res["R"], linewidth=1.5, label=label)
ax.set_xlabel("Time")
ax.set_ylabel("R")
ax.set_ylim(0, 1.05)
ax.set_title("Order Parameter R(t) -- Kernel Comparison")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, (label, res) in zip(np.ravel(axes), results_kern.items()):
    grid = res["grid"]
    plot_2d(grid.unflatten(res["state_list"][-1]["theta"]), variable="phase",
            ax=ax, title=f"{label}")
plt.suptitle(f"Phase plot at t = {T_END}s", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, (label, res) in zip(np.ravel(axes), results_kern.items()):
    grid = res["grid"]
    plot_2d(grid.unflatten(res["local_R"][-1]), variable="Local Order", vmin=0, vmax=1,
            ax=ax, title=f"{label}")
plt.suptitle(f"Local Order at t = {T_END}s", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# from kuramoto.analysis import gradient_and_material_maps
# from kuramoto.plotting import plot_gradient_map

# fig, axes = plt.subplots(2, 4, figsize=(22, 10))
# for ax, (label, res) in zip(np.ravel(axes), results_kern.items()):
#     grid = res["grid"]
#     state_list = res["state_list"]
#     t_list = res["t_list"]
#     maps = gradient_and_material_maps(state_list, t_list, grid, indices=[len(state_list) - 1])
#     dtheta_dr, dtheta_dc, magnitude, dtheta_dt, v_r, v_c, Dtheta_Dt = maps[0]
#     plot_gradient_map(
#         Dtheta_Dt,
#         dtheta_dr=v_r,
#         dtheta_dc=v_c,
#         ax=ax,
#         title=label,
#         subsample_quiver=4,
#     )
# plt.suptitle(rf"$|\nabla\theta|$ at t = {T_END}s", fontsize=14, y=1.02)
# plt.tight_layout()
# plt.show()

### Phase Animation -- 2x2 Kernel Comparison

In [ ]:
# Phase GIF animation (2x4 kernels)
# Uses the same subplot layout as the static phase plot above.

from matplotlib.animation import PillowWriter
from IPython.display import Image, display

# ---- animation settings ----
gif_path = "phase_kernels_over_time.gif"
max_frames = 200          # cap total frames to keep file size reasonable
fps = 15                  # playback speed

labels = list(results_kern.keys())

# Use the shortest run length across kernels (should be identical, but stay safe)
n_steps = min(len(results_kern[lbl]["state_list"]) for lbl in labels)

# Choose a stride so we don't exceed max_frames
stride = max(1, int(np.ceil(n_steps / max_frames)))
frame_indices = list(range(0, n_steps, stride))

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes_flat = np.ravel(axes)

# Hide any unused axes if fewer than 8 kernels
for ax in axes_flat[len(labels):]:
    ax.axis("off")

# NOTE: FIX REPEATED CBAR PLOTTING
def _draw_frame(step_idx: int):
    """Redraw all kernel phase panels for a given time index."""
    for ax in axes_flat[:len(labels)]:
        ax.clear()

    for ax, label in zip(axes_flat[:len(labels)], labels):
        res = results_kern[label]
        grid = res["grid"]
        theta = res["state_list"][step_idx]["theta"]
        plot_2d(grid.unflatten(theta), variable="phase", ax=ax, title=f"{label}")

    t_val = results_kern[labels[0]]["t_list"][step_idx]
    plt.suptitle(f"Phase plots at t = {t_val:.2f}s", fontsize=14, y=1.02)
    plt.tight_layout()


def _update(frame_i: int):
    step_idx = frame_indices[frame_i]
    _draw_frame(step_idx)
    return []


anim = FuncAnimation(
    fig,
    _update,
    frames=len(frame_indices),
    interval=int(1000 / fps),
    blit=False,
)

# Save GIF
writer = PillowWriter(fps=fps)
anim.save(gif_path, writer=writer)
plt.close(fig)

print(f"Saved GIF to: {gif_path} (frames={len(frame_indices)}, stride={stride})")
display(Image(filename=gif_path))
